# Affordability Check

This notebook demonstrates `JoinModule` — used here to enrich a customer frame with
expense data that lives in a separate source.

The pipeline:

1. `ExpenseEnricher` runs on the `expenses` frame to derive `total_expense` and `expense_ratio`
2. `JoinModule` merges the enriched expense data back onto the customer frame on `customer_id`
3. `AffordabilityScorer` computes `disposable_income` and a boolean `can_afford` flag

Both input frames are passed together as `{"input": customers_df, "expenses": expenses_df}`.

In [ ]:
%pip install -q decider


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import polars as pl

In [ ]:
from decider.modules.functional import generate_from_functions
from decider.modules._ext import register_graph_module, GraphModule
from decider.modules.primitives.join import JoinModule
from decider.modules.primitives.sequential import SequentialModule
from pydantic import BaseModel

## Sample data

Two separate frames that share `customer_id` as a join key.

In [ ]:
customers = pl.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004"],
    "income":      [48_000, 32_000, 75_000, 28_000],
    "loan_amount": [12_000,  8_000, 30_000,  5_000],
})

expenses = pl.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004"],
    "income":      [48_000, 32_000, 75_000, 28_000],
    "rent":        [ 1_200,    900,  2_500,    750],
    "food":        [   400,    300,    600,    250],
    "transport":   [   150,    100,    200,     80],
})

print("Customers:")
print(customers)
print("\nExpenses:")
print(expenses)

## Module 1 — ExpenseEnricher

Runs on the `expenses` frame.  Aggregates individual cost lines into `total_expense`
and computes `expense_ratio` as a proportion of annual income.

In [ ]:
def total_expense(rent: pl.Expr, food: pl.Expr, transport: pl.Expr) -> pl.Expr:
    return (rent + food + transport) * 12

def expense_ratio(total_expense: pl.Expr, income: pl.Expr) -> pl.Expr:
    return (total_expense / income).round(4)

ExpenseEnricher = generate_from_functions("expense_enricher", total_expense, expense_ratio)
register_graph_module(ExpenseEnricher)
expense_enricher = ExpenseEnricher(name="expense_enricher")

print(expense_enricher({"input": expenses}))

## JoinModule — merge enriched expenses back to customers

`JoinModule` accepts module references as `left` or `right`, so we can point `right`
directly at `expense_enricher` rather than pre-computing a separate frame.

In [ ]:
from decider.modules.primitives.join import FrameRef

# Route the "expenses" frame into expense_enricher using FrameRef
expense_pipeline = FrameRef(name="expenses") | expense_enricher

joiner = JoinModule(
    name="expense_join",
    left="input",
    right=expense_pipeline,
    on="customer_id",
    how="left",
)

joined = joiner({"input": customers, "expenses": expenses})
print(joined)


## Module 2 — AffordabilityScorer

Runs on the joined frame.  `disposable_income` is what is left after all annual expenses.
`can_afford` is `True` when the disposable income covers at least one monthly loan repayment.

In [ ]:
def disposable_income(income: pl.Expr, total_expense: pl.Expr) -> pl.Expr:
    return income - total_expense

def can_afford(disposable_income: pl.Expr, loan_amount: pl.Expr) -> pl.Expr:
    return disposable_income > (loan_amount / 12)

AffordabilityScorer = generate_from_functions("affordability_scorer", disposable_income, can_afford)
register_graph_module(AffordabilityScorer)
affordability_scorer = AffordabilityScorer(name="affordability_scorer")

## Full pipeline

The final pipeline is `joiner | affordability_scorer`.
Both input frames are supplied at call time — the `JoinModule` internally routes
the `"expenses"` frame through `ExpenseEnricher` before the join.

In [ ]:
pipeline = joiner | affordability_scorer

result = pipeline({"input": customers, "expenses": expenses})
print(result.select(["customer_id", "income", "loan_amount", "total_expense", "expense_ratio", "disposable_income", "can_afford"]))

## Config roundtrip

The whole pipeline, including the embedded `ExpenseEnricher` inside the `JoinModule`, serialises correctly.

In [ ]:
import json

cfg = pipeline.model_dump()
print(json.dumps(cfg, indent=2))

restored = GraphModule.model_validate(cfg).root
restored_result = restored({"input": customers, "expenses": expenses})
print("\nOutputs match after roundtrip:", result.equals(restored_result))